(Scraping and Cleaning Data)Task-1,2,3

In [1]:
# Cell 1: Web Scraping and Data Cleaning
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3

# ---------------------------------------------------------
# TASK 1: Scrape Data
# ---------------------------------------------------------
base_url = "http://books.toscrape.com/catalogue/"
books_data = []

print("Scraping in progress... (this takes a moment to fetch all pages)")

# Scraping the first 5 pages of the "All products" catalog (100 books total)
for page in range(1, 6):
    url = f"{base_url}page-{page}.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find every book container on the page
    for pod in soup.find_all('article', class_='product_pod'):
        title = pod.h3.a['title']
        price_raw = pod.find('p', class_='price_color').text
        star_rating_raw = pod.find('p', class_='star-rating')['class'][1]

        # To get the category and availability, we must visit the book's detail page
        detail_link = pod.h3.a['href']
        detail_resp = requests.get(base_url + detail_link)
        detail_soup = BeautifulSoup(detail_resp.text, 'html.parser')

        category = detail_soup.find('ul', class_='breadcrumb').find_all('li')[2].text.strip()
        availability_raw = detail_soup.find('p', class_='instock availability').text.strip()

        books_data.append({
            'title': title,
            'price': price_raw,
            'star_rating': star_rating_raw,
            'availability': availability_raw,
            'category': category
        })

df = pd.DataFrame(books_data)
print(f"Scraping complete. Total books scraped: {len(df)}")

# ---------------------------------------------------------
# TASK 2: Clean the Data
# ---------------------------------------------------------
# 1. Strip currency symbol and extract numerical price
df['price_gbp'] = df['price'].str.extract(r'([\d.]+)').astype(float)

# 2. Convert text rating to integer
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df['rating'] = df['star_rating'].map(rating_map)

# 3. Parse availability into a boolean column
df['in_stock'] = df['availability'].str.contains('In stock', case=False, na=False)

# 4. Handle failed parsing (Drop rows)
# Justification: In a pricing pipeline, imputing fake prices or ratings compromises
# the integrity of competitive analysis. Dropping invalid rows ensures data fidelity.
df.dropna(subset=['price_gbp', 'rating', 'in_stock'], inplace=True)

# ---------------------------------------------------------
# TASK 3: Currency Conversion
# ---------------------------------------------------------
df['price_inr'] = df['price_gbp'] * 105.50

print("\nSample of Cleaned Data:")
display(df[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category']].head())

Scraping in progress... (this takes a moment to fetch all pages)
Scraping complete. Total books scraped: 100

Sample of Cleaned Data:


,title,price_gbp,price_inr,rating,in_stock,category
0,A Light in the Attic,51.77,5461.735,3,True,Poetry
1,Tipping the Velvet,53.74,5669.570,1,True,Historical Fiction
2,Soumission,50.10,5285.550,1,True,Fiction
3,Sharp Objects,47.82,5045.010,4,True,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5721.265,5,True,History


Database Schema & Execution (Tasks 4 & 5)

In [2]:
# Cell 2: Database Schema, Insertion, and Queries
conn = sqlite3.connect('zepto_pipeline.db')
cursor = conn.cursor()

# ---------------------------------------------------------
# TASK 4: Normalized Schema Creation
# ---------------------------------------------------------
cursor.executescript('''
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE
);

CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
);
''')

# ---------------------------------------------------------
# TASK 5: Insert Data & Execute SQL Queries
# ---------------------------------------------------------
# Insert distinct categories
unique_categories = df[['category']].drop_duplicates()
unique_categories.to_sql('categories_temp', conn, if_exists='replace', index=False)
cursor.execute('''
    INSERT OR IGNORE INTO categories (category_name)
    SELECT category FROM categories_temp;
''')
cursor.execute('DROP TABLE categories_temp')

# Map category_id back to the books dataframe and insert books
categories_db = pd.read_sql('SELECT * FROM categories', conn)
df_merged = df.merge(categories_db, left_on='category', right_on='category_name')

books_insert = df_merged[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']].copy()
books_insert['in_stock'] = books_insert['in_stock'].astype(int) # SQLite requires 1/0 for boolean
books_insert.to_sql('books', conn, if_exists='append', index=False)

# Define and execute the 5 required SQL queries
queries = {
    "1. DISTINCT and ORDER BY": "SELECT DISTINCT category_name FROM categories ORDER BY category_name LIMIT 5;",
    "2. SELECT/WHERE and LIMIT": "SELECT title, price_gbp FROM books WHERE rating = 5 LIMIT 3;",
    "3. IN clause": "SELECT title, rating FROM books WHERE rating IN (4, 5) LIMIT 3;",
    "4. BETWEEN clause": "SELECT title, price_inr FROM books WHERE price_gbp BETWEEN 20 AND 30 LIMIT 3;",
    "5. JOIN query": """
        SELECT c.category_name, b.title, b.rating
        FROM books b
        JOIN categories c ON b.category_id = c.category_id
        LIMIT 3;
    """
}

print("--- SQL QUERY OUTPUTS ---")
for description, query in queries.items():
    print(f"\n{description}:\n{query}")
    display(pd.read_sql(query, conn))

--- SQL QUERY OUTPUTS ---

1. DISTINCT and ORDER BY:
SELECT DISTINCT category_name FROM categories ORDER BY category_name LIMIT 5;


,category_name
0,Add a comment
1,Art
2,Business
3,Childrens
4,Contemporary



2. SELECT/WHERE and LIMIT:
SELECT title, price_gbp FROM books WHERE rating = 5 LIMIT 3;


,title,price_gbp
0,Sapiens: A Brief History of Humankind,54.23
1,Set Me Free,17.46
2,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29



3. IN clause:
SELECT title, rating FROM books WHERE rating IN (4, 5) LIMIT 3;


,title,rating
0,Sharp Objects,4
1,Sapiens: A Brief History of Humankind,5
2,The Dirty Little Secrets of Getting Your Dream...,4



4. BETWEEN clause:
SELECT title, price_inr FROM books WHERE price_gbp BETWEEN 20 AND 30 LIMIT 3;


,title,price_inr
0,The Requiem Red,2389.575
1,The Boys in the Boat: Nine Americans and Their...,2384.300
2,Shakespeare's Sonnets,2179.630



5. JOIN query:

        SELECT c.category_name, b.title, b.rating 
        FROM books b 
        JOIN categories c ON b.category_id = c.category_id 
        LIMIT 3;
    


,category_name,title,rating
0,Poetry,A Light in the Attic,3
1,Historical Fiction,Tipping the Velvet,1
2,Fiction,Soumission,1


Pandas vs SQL Comparison (Task 6)

In [3]:
# Cell 3: Comparing SQL JOIN vs Pandas .merge()

# Read back using SQL
sql_join_query = """
SELECT c.category_name, b.title, b.rating
FROM books b
JOIN categories c ON b.category_id = c.category_id
LIMIT 5
"""
df_sql_output = pd.read_sql(sql_join_query, conn)

# Read raw tables and reproduce using Pandas
df_books_raw = pd.read_sql('SELECT * FROM books', conn)
df_cats_raw = pd.read_sql('SELECT * FROM categories', conn)

df_pandas_output = pd.merge(df_books_raw, df_cats_raw, on='category_id')
# Select and reorder columns to match SQL output
df_pandas_output = df_pandas_output[['category_name', 'title', 'rating']].head(5)

print("--- SQL JOIN OUTPUT ---")
display(df_sql_output)
print("\n--- PANDAS .merge() OUTPUT ---")
display(df_pandas_output)

# Close the database connection
conn.close()

--- SQL JOIN OUTPUT ---


,category_name,title,rating
0,Poetry,A Light in the Attic,3
1,Historical Fiction,Tipping the Velvet,1
2,Fiction,Soumission,1
3,Mystery,Sharp Objects,4
4,History,Sapiens: A Brief History of Humankind,5



--- PANDAS .merge() OUTPUT ---


,category_name,title,rating
0,Poetry,A Light in the Attic,3
1,Historical Fiction,Tipping the Velvet,1
2,Fiction,Soumission,1
3,Mystery,Sharp Objects,4
4,History,Sapiens: A Brief History of Humankind,5
